# Day 3: Hypothesis testing
## Bioinformatics Course - IMBB-FORTH

### Objectives

- Understand what a p-value means by building one from scratch
- Use permutation tests to test a hypothesis
- Apply t-test and Mann-Whitney U test
- Measure and test correlations between variables

---

## 1. Setup

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats

sns.set_theme(style="whitegrid")

beaches = pd.read_csv('../data/cretan_beaches.csv')
print("Loaded", beaches.shape[0], "beaches,", beaches.shape[1], "variables")

ImportError: Unable to import required dependencies:
numpy: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

---

## 2. Question

We saw that **north-coast beaches** appear to be warmer than **south-coast beaches**.

Is this a real pattern, or could it happen by chance?

This is the fundamental question behind every statistical test.

### Step 1: Look at the data

In [3]:
# Split beaches into north and south groups
north = beaches[beaches['compass_direction'].isin(['N', 'NE', 'NW'])]
south = beaches[beaches['compass_direction'].isin(['S', 'SW', 'SE'])]

print("North coast:", len(north), "beaches, mean temp =", round(north['sea_temp_celsius'].mean(), 1), "°C")
print("South coast:", len(south), "beaches, mean temp =", round(south['sea_temp_celsius'].mean(), 1), "°C")
print("Difference:", round(north['sea_temp_celsius'].mean() - south['sea_temp_celsius'].mean(), 1), "°C")

NameError: name 'beaches' is not defined

In [4]:
# Visualize the two distributions
sns.boxplot(data=beaches, 
            x=beaches['compass_direction'].isin(['N', 'NE', 'NW']).map({True: 'North', False: 'South'}),
            y='sea_temp_celsius')
print("The difference looks real — but how can we be sure?")

NameError: name 'sns' is not defined

---

## 3. Understanding P-values: The Permutation Test

The idea is simple:

1. We observed a temperature difference of ~3°C between north and south.
2. **What if the labels "north" and "south" don't matter?** Then any beach could be in either group.
3. We shuffle the labels randomly and recalculate the difference.
4. We do this thousands of times.
5. **How often do we get a difference as large as what we actually observed?**

That proportion is the **p-value**.

### Step 2: Calculate the observed difference

In [5]:
# Our real observed difference
north_temps = north['sea_temp_celsius'].values
south_temps = south['sea_temp_celsius'].values

observed_diff = north_temps.mean() - south_temps.mean()
print("Observed difference:", round(observed_diff, 2), "°C")

NameError: name 'north' is not defined

### Step 3: Shuffle and repeat

If north/south labels are meaningless, we can shuffle them.
Each shuffle gives us a "fake" difference that could happen by chance.

In [6]:
# Combine all temperatures into one pool
all_temps = list(north_temps) + list(south_temps)
all_temps = np.array(all_temps)  # convert back for shuffling
n_north = len(north_temps)

# One shuffle: randomly assign labels
shuffled = np.random.permutation(all_temps)
fake_north = shuffled[:n_north]
fake_south = shuffled[n_north:]
fake_diff = fake_north.mean() - fake_south.mean()

print("One random shuffle gives a difference of:", round(fake_diff, 2), "°C")
print("Our real difference was:", round(observed_diff, 2), "°C")

NameError: name 'north_temps' is not defined

### Step 4: Do this 10,000 times

In [7]:
# Permutation test: shuffle 10,000 times
np.random.seed(42)
n_permutations = 10000
shuffled_diffs = []

for i in range(n_permutations):
    shuffled = np.random.permutation(all_temps)
    fake_diff = shuffled[:n_north].mean() - shuffled[n_north:].mean()
    shuffled_diffs.append(fake_diff)

shuffled_diffs = np.array(shuffled_diffs)
print("Generated", n_permutations, "random differences")
print("Range:", round(shuffled_diffs.min(), 2), "to", round(shuffled_diffs.max(), 2), "°C")

NameError: name 'np' is not defined

### Step 5: Where does our real observation fall?

This is the key plot. The histogram shows what differences look like **under pure chance**.
The red line shows what we **actually observed**.

In [8]:
# Plot the distribution of random differences
sns.histplot(shuffled_diffs, bins=50, color='steelblue')

# Mark our observed difference
import matplotlib.pyplot as plt
plt.axvline(observed_diff, color='red', linewidth=2, label='Observed: ' + str(round(observed_diff, 2)) + '°C')
plt.xlabel('Difference in mean temperature (°C)')
plt.ylabel('Count')
plt.legend()
plt.show()

print("The red line is far to the right — our observation is very unusual under random chance.")

NameError: name 'sns' is not defined

### Step 6: Calculate the p-value

In [9]:
# P-value: how often is a random difference >= our observed difference?
p_value = np.sum(np.abs(shuffled_diffs) >= np.abs(observed_diff)) / n_permutations

print("Observed difference:", round(observed_diff, 2), "°C")
print("P-value:", p_value)
print()
if p_value < 0.05:
    print("p < 0.05: The difference is statistically significant.")
    print("It would be very unlikely to see this difference if north/south didn't matter.")
else:
    print("p >= 0.05: The difference is not statistically significant.")
    print("A difference this large could easily happen by chance.")

NameError: name 'np' is not defined

### What the p-value means

The p-value answers:

> **If there were no real difference, how likely would we be to see a result as extreme as this?**

A small p-value (< 0.05) means our observation is **rare under the assumption of no difference**.
It does not prove the difference is real — it says it would be surprising if it weren't.

---

## 4. Standard Statistical Tests

The permutation test builds intuition. In practice, there are established tests that give similar answers.

### T-test (parametric)

Assumes data are roughly normally distributed. Tests whether two group means differ.

In [10]:
# T-test: are north and south temperatures different?
t_stat, p_ttest = stats.ttest_ind(north_temps, south_temps)

print("T-statistic:", round(t_stat, 2))
print("P-value:", round(p_ttest, 6))

NameError: name 'stats' is not defined

### Mann-Whitney U test (non-parametric)

Does not assume normal distributions. Tests whether one group tends to have larger values.

In [11]:
# Mann-Whitney U: non-parametric alternative
u_stat, p_mann = stats.mannwhitneyu(north_temps, south_temps, alternative='two-sided')

print("U-statistic:", round(u_stat, 0))
print("P-value:", round(p_mann, 6))

NameError: name 'stats' is not defined

### Comparison

All three methods agree:

In [12]:
# Compare all three results
print("Method               P-value       Significant?")
print("-" * 48)
print("Permutation test    ", round(p_value, 4), "       ", "Yes" if p_value < 0.05 else "No")
print("T-test              ", round(p_ttest, 4), "       ", "Yes" if p_ttest < 0.05 else "No")
print("Mann-Whitney U      ", round(p_mann, 4), "       ", "Yes" if p_mann < 0.05 else "No")

Method               P-value       Significant?
------------------------------------------------


NameError: name 'p_value' is not defined

### 💡 Exercise 3.1: Test Another Comparison

**Task:** Do sandy beaches attract more tourists than pebble beaches?

1. Get tourist numbers for sand and pebble beaches
2. Calculate the observed difference in means
3. Run a t-test
4. Is the difference significant?

In [13]:
# Your code here
sand_tourists = beaches[beaches['beach_type'] == 'sand']['avg_tourists_per_day'].values
pebble_tourists = beaches[beaches['beach_type'] == 'pebble']['avg_tourists_per_day'].values

print("Sand mean:", round(sand_tourists.mean()), "tourists/day, n =", len(sand_tourists))
print("Pebble mean:", round(pebble_tourists.mean()), "tourists/day, n =", len(pebble_tourists))

# Run t-test
t, p = # Your code
print("T-test p-value:", round(p, 4))

SyntaxError: invalid syntax (587781658.py, line 9)

---

## 5. Correlation

Correlation measures how two variables change together.

**Pearson correlation (r)** ranges from -1 to +1:
- **r = +1**: perfect positive relationship (both increase together)
- **r = 0**: no linear relationship
- **r = -1**: perfect negative relationship (one increases, the other decreases)

### Temperature vs Tourism

In [14]:
# Scatter plot with correlation
sns.scatterplot(data=beaches, x='sea_temp_celsius', y='avg_tourists_per_day')

r, p = stats.pearsonr(beaches['sea_temp_celsius'], beaches['avg_tourists_per_day'])
print("Pearson r =", round(r, 3))
print("P-value =", round(p, 4))
print("Strong positive correlation: warmer beaches attract more tourists")

NameError: name 'sns' is not defined

### Lionfish vs Cornetfish

In [15]:
# Two invasive species: do they co-occur?
sns.scatterplot(data=beaches, x='lionfish_abundance', y='cornetfish_abundance')

r, p = stats.pearsonr(beaches['lionfish_abundance'], beaches['cornetfish_abundance'])
print("Pearson r =", round(r, 3))
print("P-value =", round(p, 4))
print("Strong positive correlation: these species tend to invade the same beaches")

NameError: name 'sns' is not defined

### Rainfall vs Longitude (West-East gradient)

In [16]:
# Does rainfall decrease from west to east Crete?
sns.scatterplot(data=beaches, x='longitude', y='annual_rainfall_mm')

r, p = stats.pearsonr(beaches['longitude'], beaches['annual_rainfall_mm'])
print("Pearson r =", round(r, 3))
print("P-value =", round(p, 4))
print("Strong negative correlation: western Crete is wetter than eastern Crete")

NameError: name 'sns' is not defined

### Correlation Heatmap

We can visualize all pairwise correlations at once.

In [17]:
# Select numerical columns for correlation
num_cols = ['longitude', 'beach_length_m', 'phoenix_palms', 'juniper_density',
            'lionfish_abundance', 'cornetfish_abundance',
            'sea_temp_celsius', 'wind_speed_kmh', 'annual_rainfall_mm', 'avg_tourists_per_day']

corr_matrix = beaches[num_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
print("Red = positive correlation, Blue = negative correlation")

NameError: name 'beaches' is not defined

### 💡 Exercise 3.2: Explore Correlations

**Task:**
1. Test the correlation between wind speed and tourism. Is it significant?
2. Test the correlation between beach length and juniper density.
3. Look at the heatmap. Which pair of variables has the strongest negative correlation?

In [18]:
# 1. Wind speed vs tourism
r, p = # Your code
print("Wind vs tourism: r =", round(r, 3), ", p =", round(p, 4))

# 2. Beach length vs juniper density
r, p = # Your code
print("Length vs juniper: r =", round(r, 3), ", p =", round(p, 4))

# 3. Strongest negative correlation from the heatmap?
# Your answer: 

SyntaxError: invalid syntax (3193822661.py, line 2)

---

## 6. Visualize First

A key habit: **look at your data before running any test.**

Let's compare north and south temperature distributions properly.

In [19]:
# Overlapping histograms
sns.histplot(north['sea_temp_celsius'], color='coral', label='North', kde=True, alpha=0.5)
sns.histplot(south['sea_temp_celsius'], color='steelblue', label='South', kde=True, alpha=0.5)
plt.xlabel('Sea Temperature (°C)')
plt.legend()
plt.show()

print("The distributions barely overlap — the difference is clear even before any test.")

NameError: name 'sns' is not defined

---

## 💡 Final Exercise: Full Analysis

**Task:** Is there a difference in lionfish abundance between north-coast and south-coast beaches?

1. Calculate mean lionfish for each group
2. Visualize the two distributions
3. Run a permutation test (1000 shuffles is enough)
4. Run a t-test
5. What do you conclude?

In [ ]:
# Your code here

# 1. Group means
north_lion = # Your code
south_lion = # Your code

print("North mean lionfish:", round(north_lion.mean(), 1))
print("South mean lionfish:", round(south_lion.mean(), 1))

# 2. Visualize
# Your code

# 3. Permutation test
# Your code

# 4. T-test
# Your code

# 5. Conclusion?


---

## Summary

### What is a p-value?

The p-value tells you: **if there were no real effect, how often would we find a result as extreme as what we observed by random chance**

### Core Concepts
- **Permutation test** — estimate significance by shuffling labels and measuring how extreme your observation is
- **T-test** — parametric test for difference in means
- **Mann-Whitney U** — non-parametric test (no normality assumption)
- **Pearson correlation** — measures linear relationship strength (r) and significance (p)

### Key Functions
- `stats.ttest_ind(group1, group2)` — independent samples t-test
- `stats.mannwhitneyu(group1, group2)` — Mann-Whitney U test
- `stats.pearsonr(x, y)` — Pearson correlation with p-value

### Key Take-home

1. **Visualize first, test second**
2. The p-value is not the probability that your hypothesis is true
3. Statistical significance is not the same as biological importance
4. Effect size matters — a tiny difference can be "significant" with enough data

### Tomorrow: Day 4

**enrichment analysis**:
- Are certain categories over-represented in a subset?
- Same permutation logic, different application
- Then: GO terms and differential expression